# IJSON Demo

This demo shows basic use cases for ijson and how simple changes can influence performance

In [1]:
import json
import ijson
import tracemalloc
import os

## 1. Create a sample JSON file

In [ ]:
def create_sample_file(filename, num_records=200_000):
    print(f"Creating {filename} with {num_records} records...")
    with open(filename, "w") as f:
        f.write("[")
        for i in range(num_records):
            record = {"id": i, "text": f"Sample sentence number {i}"}
            f.write(json.dumps(record))
            if i < num_records - 1:
                f.write(",")
        f.write("]")
    file_size = os.path.getsize(filename) / (1024 * 1024)
    print(f"File created. Size: {file_size:.2f} MB\n")

## 2. Memory efficient processing

In [ ]:
def process_with_ijson(filename):
    tracemalloc.start()
    peak_before = tracemalloc.get_traced_memory()[1] / (1024 * 1024)

    count = 0
    with open(filename, "rb") as f:
        # Generator: yields one item at a time, never loads whole list
        for record in ijson.items(f, "item"):
            # Simulate processing (e.g., extract field)
            _ = record["text"]
            count += 1
            # Stop early to keep demo fast
            if count >= 100_000:
                break

    current, peak = tracemalloc.get_traced_memory()
    peak_mb = peak / (1024 * 1024)
    tracemalloc.stop()
    print(f"[ijson] Processed {count} records")
    print(f"[ijson] Peak memory usage: {peak_mb:.2f} MB")
    return peak_mb

## 3. Memory inefficient processing

In [ ]:
def process_with_json_load(filename):
    tracemalloc.start()
    peak_before = tracemalloc.get_traced_memory()[1] / (1024 * 1024)

    with open(filename, "r") as f:
        data = json.load(f)   # Loads the entire array into memory

    count = 0
    for record in data:
        _ = record["text"]
        count += 1
        if count >= 100_000:
            break

    current, peak = tracemalloc.get_traced_memory()
    peak_mb = peak / (1024 * 1024)
    tracemalloc.stop()
    print(f"[json.load] Processed {count} records")
    print(f"[json.load] Peak memory usage: {peak_mb:.2f} MB")
    return peak_mb

## Comparison

In [ ]:
filename = "large_data.json"
    
# Create file only once
create_sample_file(filename, num_records=200_000)

print("=" * 50)
print("Running ijson (streaming) version...")
mem_ijson = process_with_ijson(filename)

print("\n" + "=" * 50)
print("Running json.load() (full load) version...")
mem_jsonload = process_with_json_load(filename)

print("\n" + "=" * 50)
print(f"Memory saved with ijson: {mem_jsonload - mem_ijson:.2f} MB")